In [4]:
import time
import json
import hashlib
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from mineru import DoclibClient
from mineru.doclib import ParseRequest

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
MARKDOWN_DIR = PROJECT_ROOT / "data" / "processed" / "markdown"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(RAW_DIR.glob("*.pdf"))

assert len(pdf_files) == 5, (
    f"Expected 5 PDFs in {RAW_DIR}, found {len(pdf_files)}"
)

display(pd.DataFrame([
    {
        "filename": p.name,
        "size_mb": round(p.stat().st_size / (1024 ** 2), 2),
    }
    for p in pdf_files
]))

,filename,size_mb
0,AGI.pdf,7.03
1,Foundation_Models.pdf,13.75
2,GPT-4.pdf,5.00
3,Holistic_Eval.pdf,5.77
4,LLMs.pdf,5.58


In [8]:
client = DoclibClient()

test_pdf = pdf_files[0]

submit = client.ensure_parse(
    ParseRequest(
        path=str(test_pdf.resolve()),
        tier="flash",
    )
)

print("Document:", test_pdf.name)
print("Document ID:", submit.short_id)
print("Parse task IDs:", submit.wait_parse_ids)

Document: AGI.pdf
Document ID: dee9263
Parse task IDs: []


In [9]:
def wait_for_parse(parse_id, poll_interval=2):
    parse = client.get_parse(parse_id)

    while parse.status in ("pending", "parsing"):
        time.sleep(poll_interval)
        parse = client.get_parse(parse_id)

    if parse.status != "done":
        raise RuntimeError(
            f"Parse {parse_id} ended as {parse.status}: "
            f"{parse.error_code} {parse.error_msg}"
        )

    return parse


completed_tasks = []

for parse_id in submit.wait_parse_ids:
    parse = wait_for_parse(parse_id)
    completed_tasks.append(parse_id)

print("Completed parse tasks:", completed_tasks)

Completed parse tasks: []


In [16]:
def save_document_markdown(short_id, pdf_path):
    """
    Read parsed pages and save their content as one Markdown file.

    Assumes the parsed document uses sequential page numbers
    starting at 1, as in the Doclib content API.
    """
    output_path = MARKDOWN_DIR / f"{pdf_path.stem}.md"

    page_number = 1
    page_sections = []

    while True:
        try:
            content = client.read_content(
                f"doc:{short_id}/tier:flash/page:{page_number}"
            )
        except Exception as exc:
            # A missing page ends the sequence only after at
            # least one page was successfully retrieved.
            if page_sections:
                print(
                    f"Stopped at page number {page_number}: {exc}"
                )
                break
            raise

        page_content = content.content

        if not page_content:
            break

        page_sections.append(
            f"<!-- Source page: {page_number} -->\n\n"
            f"{page_content}"
        )

        page_number += 1

    if not page_sections:
        raise RuntimeError(
            f"No page content retrieved for {pdf_path.name}"
        )

    output_path.write_text(
        "\n\n---\n\n".join(page_sections),
        encoding="utf-8",
    )

    return {
        "document": pdf_path.name,
        "pages_read": len(page_sections),
        "characters": sum(map(len, page_sections)),
        "markdown_path": str(output_path),
        "status": "success",
    }


test_result = save_document_markdown(
    submit.short_id,
    test_pdf,
)

display(pd.DataFrame([test_result]))

Stopped at page number 11: Requested parsed content is not cached.


,document,pages_read,characters,markdown_path,status
0,AGI.pdf,10,38282,c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\d...,success


In [ ]:
manifest = []

for pdf in tqdm(pdf_files, desc="Processing AI/ML PDFs"):
    output_path = MARKDOWN_DIR / f"{pdf.stem}.md"

    if output_path.exists() and output_path.stat().st_size > 0:
        manifest.append({
            "document": pdf.name,
            "status": "skipped_existing",
            "markdown_path": str(output_path),
        })
        continue

    try:
        submission = client.ensure_parse(
            ParseRequest(
                path=str(pdf.resolve()),
                tier="standard",
            )
        )

        for parse_id in submission.wait_parse_ids:
            wait_for_parse(parse_id)

        result = save_document_markdown(
            submission.short_id,
            pdf,
        )

        manifest.append(result)

    except Exception as exc:
        manifest.append({
            "document": pdf.name,
            "status": "failed",
            "error": str(exc),
        })

    (ARTIFACTS_DIR / "ingestion_manifest.json").write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

display(pd.DataFrame(manifest))

In [ ]:
inspection = []

for md_file in sorted(MARKDOWN_DIR.glob("*.md")):
    text = md_file.read_text(encoding="utf-8")
    lines = text.splitlines()

    inspection.append({
        "filename": md_file.name,
        "characters": len(text),
        "words": len(text.split()),
        "headings": sum(
            line.lstrip().startswith("#")
            for line in lines
        ),
        "table_lines": sum(
            "|" in line for line in lines
        ),
        "empty": not text.strip(),
        "preview": " ".join(text.split())[:200],
    })

inspection_df = pd.DataFrame(inspection)
display(inspection_df)